# W8C2 Lab: BERT in code

Run every cell from the top. **Everything already works.**

DistilBERT is downloaded once (about 260 MB) and cached after that. Everything here runs on a laptop CPU.

Today you will:

1. Build the input BERT actually reads.
2. Ask it to fill in a hidden word, and read the numbers behind the guess.
3. Use the `[CLS]` vector as a sentence, then fine-tune the whole model.

Nothing to submit. Answers are in the last cell.

In [ ]:
# Setup. Run this cell first.
import warnings
import time
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(NAME)

headlines = pd.read_csv("data/headlines.csv").sample(4000, random_state=0)
print(f"{len(headlines)} headlines, {headlines.is_sarcastic.mean():.0%} sarcastic")

## Part 1. What the model actually reads

A sentence pair goes in as three rows of numbers: which word, which sentence, and which positions are real.

In [ ]:
batch = tokenizer("the bank was closed", "we walked home",
                  return_token_type_ids=True)

print(pd.DataFrame({
    "token": tokenizer.convert_ids_to_tokens(batch["input_ids"]),
    "input_id": batch["input_ids"],
    "segment": batch["token_type_ids"],
    "attention_mask": batch["attention_mask"],
}).to_string(index=False))

In [ ]:
# Padding is what the attention mask is for: 0 means "not a real token".
padded = tokenizer(["short one", "a considerably longer headline than that one"],
                   padding=True)
for ids, mask in zip(padded["input_ids"], padded["attention_mask"]):
    print(tokenizer.convert_ids_to_tokens(ids), mask)

In [ ]:
# ================== TRY IT 1 ==================
# Tokenize a sentence with an unusual word in it, such as
# "the paleontologist tokenized it". Which words survive whole?
# ==============================================


## Part 2. Fill in a hidden word

The pretraining task, run directly: hide a token, and read the distribution the model puts over the vocabulary at that position.

In [ ]:
mlm = AutoModelForMaskedLM.from_pretrained(NAME)
mlm.eval()


def predictions_at_mask(text, k=5):
    """The k most likely words for the single [MASK] in `text`."""
    batch = tokenizer(text, return_tensors="pt")
    position = (batch["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()
    with torch.no_grad():
        logits = mlm(**batch).logits[0, position]
    probability = logits.softmax(dim=-1)
    top = probability.topk(k)
    words = tokenizer.convert_ids_to_tokens(top.indices)
    return pd.DataFrame({"word": words, "probability": top.values.numpy().round(3)})


print(predictions_at_mask("i deposited money at the [MASK].").to_string(index=False))

In [ ]:
# The same sentence with one word changed picks a different answer.
print(predictions_at_mask("i sat and watched the river [MASK].").to_string(index=False))

In [ ]:
def loss_for(text, word):
    """Cross entropy at the masked position if `word` were the original."""
    batch = tokenizer(text, return_tensors="pt")
    position = (batch["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()
    with torch.no_grad():
        logits = mlm(**batch).logits[0, position]
    target = tokenizer.convert_tokens_to_ids(word)
    return float(torch.nn.functional.cross_entropy(logits[None], torch.tensor([target])))


sentence = "i deposited money at the [MASK]."
for word in ["bank", "shore", "table"]:
    print(f"{word:8s} loss {loss_for(sentence, word):.2f}")

In [ ]:
# ================== TRY IT 2 ==================
# Write a sentence where the model's top guess is wrong.
# What does its mistake tell you about the text it was trained on?
# ==============================================


## Part 3. One vector for a whole sentence

The final vector at `[CLS]` is a summary of the sentence. Freeze the model, take that vector, and train a plain classifier on top.

In [ ]:
encoder = AutoModel.from_pretrained(NAME)
encoder.eval()


def cls_vectors(texts, size=64):
    """The [CLS] vector for each text, from the frozen encoder."""
    rows = []
    with torch.no_grad():
        for start in range(0, len(texts), size):
            batch = tokenizer(texts[start:start + size], padding=True,
                              truncation=True, max_length=32, return_tensors="pt")
            rows.append(encoder(**batch).last_hidden_state[:, 0])
    return torch.cat(rows).numpy()


train, test = headlines.iloc[:3000], headlines.iloc[3000:]

started = time.time()
train_x = cls_vectors(train.headline.tolist())
test_x = cls_vectors(test.headline.tolist())
print(f"{train_x.shape[1]} numbers per headline, {time.time() - started:.0f}s")

In [ ]:
frozen = LogisticRegression(max_iter=2000).fit(train_x, train.is_sarcastic)
BASELINE = frozen.score(test_x, test.is_sarcastic)
print(f"frozen [CLS] + logistic regression: {BASELINE:.3f}")

In [ ]:
# ================== TRY IT 3 ==================
# Average all the token vectors instead of taking [CLS] alone.
# Does the classifier do better?
# ==============================================


---

## After the break: beat the frozen baseline

The encoder above never learned anything about sarcasm; only the classifier on
top did. Fine-tuning trains the whole thing.

In [ ]:
from transformers import AutoModelForSequenceClassification


def fresh_model():
    """A pretrained encoder with a new classifier on top. Seeded, so two runs
    of the same settings give the same number."""
    torch.manual_seed(0)
    return AutoModelForSequenceClassification.from_pretrained(NAME, num_labels=2)


def batches(frame, size):
    for start in range(0, len(frame), size):
        chunk = frame.iloc[start:start + size]
        yield (tokenizer(chunk.headline.tolist(), padding=True, truncation=True,
                         max_length=32, return_tensors="pt"),
               torch.tensor(chunk.is_sarcastic.values))


def train_and_score(model, rows, epochs, learning_rate):
    """Train on the first `rows` headlines, then score on the held out 1000."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    model.train()
    for _ in range(epochs):
        for batch, labels in batches(train.iloc[:rows], 16):
            optimizer.zero_grad()
            model(**batch, labels=labels).loss.backward()
            optimizer.step()
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch, labels in batches(test, 64):
            correct += int((model(**batch).logits.argmax(-1) == labels).sum())
    return correct / len(test)

In [ ]:
# ================== YOUR TURN 1 ==================
# Fine-tune the whole model and beat the frozen baseline.
# ROWS, EPOCHS and LEARNING_RATE are yours to change.
#
# Expected: the starting settings score 0.806, under the frozen 0.827.
#           3000 rows for 1 epoch reaches 0.876, in roughly a minute.
# =================================================
ROWS, EPOCHS, LEARNING_RATE = 300, 1, 3e-5   # <-- edit these

model = fresh_model()
started = time.time()
score = train_and_score(model, ROWS, EPOCHS, LEARNING_RATE)
print(f"fine-tuned {score:.3f}   baseline {BASELINE:.3f}   {time.time() - started:.0f}s")

In [ ]:
# ================== YOUR TURN 2 ==================
# Freeze the encoder so only the classifier on top trains, and run it again.
# Use the same ROWS, EPOCHS and LEARNING_RATE you settled on.
#
# Expected: 592,130 parameters move instead of 67 million, it runs about three
#           times faster, and it scores 0.686 against 0.876.
# =================================================
frozen_model = fresh_model()

for parameter in frozen_model.distilbert.parameters():
    parameter.requires_grad = True          # <-- edit this line

moving = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)
started = time.time()
score = train_and_score(frozen_model, ROWS, EPOCHS, LEARNING_RATE)
print(f"{moving:,} parameters training   {score:.3f}   {time.time() - started:.0f}s")

## Answers

Try each task before reading.

In [ ]:
# TRY IT 1
#   "paleontologist" becomes pale ##ont ##ologist and "tokenized" becomes
#   token ##ized. Common words are whole; rare ones are spelled out of pieces,
#   which is how a fixed vocabulary covers a word it has never seen.
#
# TRY IT 2
#   Any sentence about a rare name, a recent event, or a specialist term. The
#   model answers from the text it read, so its mistakes are a portrait of that
#   text, not of the world.
#
# TRY IT 3
#       pooled = encoder(**batch).last_hidden_state.mean(dim=1)
#   Mean pooling scores about 0.831 against 0.827 for [CLS] alone, which is a
#   tie. [CLS] is only a summary because pretraining made it one, and next
#   sentence prediction is a thin reason to trust it.
#
# YOUR TURN 1
#       ROWS, EPOCHS, LEARNING_RATE = 3000, 1, 3e-5
#   0.876 in about a minute, against 0.827 frozen. More rows is what buys the
#   gain; a second epoch at this learning rate drops it to 0.848.
#   `fresh_model` seeds the new head, so the same settings give the same
#   number twice; without that the score swings by several points per run.
#
# YOUR TURN 2
#       parameter.requires_grad = False
#   592,130 parameters move instead of 67 million, and the score falls to
#   0.686. Freezing is cheap and, on this task, not nearly enough: the sentence
#   vector has to change, not just the line drawn through it.